# Import data

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/voc-data.csv', encoding='utf-8', on_bad_lines='skip')
df.head()

,หมายเลขเสียง,ประเภทสารสนเทศ,วันที่บันทึกข้อมูล,พื้นที่ที่มีประเด็นเสียง,Unnamed: 4,Unnamed: 5,ช่องทางการรับฟังเสียง,กลไก,CA,ชื่อลูกค้า,กลุ่มลูกค้า,พื้นที่จ่ายไฟ,วันที่ได้รับข้อมูล,หัวข้อเสียง,ประเด็นเสียง,ประเภท VOC,รายละเอียดเสียง
0,NaN,NaN,NaN,สายงาน,ฝ่าย/เขต,กอง/การไฟฟ้า,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C-59720851,ความคาดหวัง,29/03/2024 10:12 น.,สายงานการไฟฟ้า ภาค 2,การไฟฟ้าเขต 2 (อุบลราชธานี) ภาค 2,E09201 : การไฟฟ้าส่วนภูมิภาคสาขาอำเภอโพนทอง จั...,Physical,1129 PEA Contact Center,NaN,สุรพล ผงทอง,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,\nการให้บริการ - ทั่วไป,ติดตามสถานะการขอใช้บริการ\n,บริการ,ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะกา...
2,M-67002090,ความต้องการ,29/03/2024 10:12 น.,NaN,NaN,NaN,Digital,PEA Mobile Application (Smart Plus),020012705638,ประภาพร แพงถิ่น,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,ความถูกต้องของข้อมูลบน Website/Application PEA,สนับสนุน,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...
3,C-59721018,ความคาดหวัง,29/03/2024 10:09 น.,ฝ่ายงานผู้ว่าการ,NaN,NaN,Physical,1129 PEA Contact Center,NaN,วีราพล N/A,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,การใช้งานบน Website/Application PEA,สนับสนุน,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: ยกเลิกเบ...
4,C-59720995,ความคาดหวัง,29/03/2024 10:07 น.,ฝ่ายงานผู้ว่าการ,NaN,NaN,Physical,1129 PEA Contact Center,NaN,ดวงใจ N/A,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,การใช้งานบน Website/Application PEA,สนับสนุน,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...


In [2]:
df.dropna(subset=['รายละเอียดเสียง'], inplace=True)
text_df = df[['หมายเลขเสียง','รายละเอียดเสียง']].copy()

In [3]:
text_df.rename(columns={'หมายเลขเสียง':'voc_id','รายละเอียดเสียง': 'text'}, inplace=True)

In [4]:
text_df.head()

,voc_id,text
1,C-59720851,ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะกา...
2,M-67002090,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...
3,C-59721018,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: ยกเลิกเบ...
4,C-59720995,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...
5,C-59720934,ร้องขอกฟอ สามชุก จังหวัดสุพรรณบุรี เรื่องสถานะ...


# Define function to doing text nane entity recognition

In [5]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from pythainlp.tokenize import word_tokenize # pip install pythainlp
import torch


In [6]:
text_df.loc[1]['text']

'ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะการขอขยายเขต ของบ้านเลขที่167 ม.13 ต.สระนกแก้ว อ.โพนทอง จ.ร้อยเอ็ด ชื่อผู้ขอจดทะเบียนใช้ไฟฟ้าทองอินทร์ ผงทอง วันที่ยื่นเรื่อง5/3/2567 ผู้ใช้ไฟฟ้าแจ้งว่าได้ไปส่งเอกสารครบทั้งหมดแล้วเจ้าหน้าที่แจ้งว่างบประมาณหมด ต้องส่งเรื่องไปการไฟฟ้าอุบลราชธานี แต่เจ้าหน้าที่ยังไม่มีการส่งเรื่องไป ผชฟ.ต้องให้การไฟฟ้าโฟนทองส่งเรื่องไปที่การไฟฟ้าอุบลราชธานีให้ เนื่องจากต้องการใช้ไฟฟ้าด่วน รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //คุณสุรพล  หมายเลขติดต่อกลับ 0986451488'

In [7]:
def get_ner_tag(sentence):
    name = "Porameht/wangchanberta-thainer-corpus-v2-2"
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForTokenClassification.from_pretrained(name)
    cut = word_tokenize(sentence.replace(" ", "<_>"))
    inputs = tokenizer(cut, is_split_into_words=True, return_tensors="pt")
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    outputs = model(ids, attention_mask=mask)
    logits = outputs[0]
    predictions = torch.argmax(logits, dim=2)
    predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

    def fix_span_error(words, ner):
        _new_tag = []
        for i, j in zip(words, ner):
            i = tokenizer.decode(i)
            if i.isspace() and j.startswith("B-"):
                j = "O"
            if i == '' or i == '<s>' or i == '</s>':
                continue
            if i == "<_>":
                i = " "
            _new_tag.append((i, j))
        return _new_tag

    ner_tag = fix_span_error(inputs['input_ids'][0], predicted_token_class)
    return ner_tag


In [8]:
text = text_df.loc[1]['text']
ner_tag = get_ner_tag(text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/365 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/419M [00:00<?, ?B/s]

In [9]:
ner_tag

[('ร้อง', 'O'),
 ('ขอ', 'O'),
 ('ก', 'O'),
 ('ฟ', 'O'),
 ('อ', 'I-ORGANIZATION'),
 ('.', 'B-LOCATION'),
 ('โพน', 'I-ORGANIZATION'),
 ('ทอง', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('จังหวัด', 'B-LOCATION'),
 ('ร้อยเอ็ด', 'I-LOCATION'),
 (' ', 'O'),
 ('เรื่อง', 'O'),
 ('สถานะ', 'O'),
 ('การ', 'O'),
 ('ขอ', 'O'),
 ('ขยาย', 'O'),
 ('เขต', 'O'),
 (' ', 'O'),
 ('ของ', 'O'),
 ('บ้าน', 'O'),
 ('เลขที่', 'I-LOCATION'),
 ('16', 'I-LOCATION'),
 ('7', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('ม', 'I-LOCATION'),
 ('.', 'I-LOCATION'),
 ('13', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('ต', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('สระ', 'I-LOCATION'),
 ('นก', 'I-LOCATION'),
 ('แก้ว', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('อ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('โพน', 'I-LOCATION'),
 ('ทอง', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('จ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('ร้อยเอ็ด', 'I-LOCATION'),
 (' ', 'O'),
 ('ชื่อ', 'O'),
 ('ผู้', 'O'),
 ('ขอ', 'O'),
 ('จดทะเบียน', 'O'),
 ('ใช้', 'O'),
 ('ไฟฟ้า', 

In [10]:
def ner_nasking(ner_tag:list):
    masked = ""
    current_entity = None 

    for token, tag in ner_tag:
        if tag == 'O':
            masked += token
            current_entity = None
        else:
            entity = tag.split('-')[1]
            if tag.startswith('B-') or current_entity != entity:
                masked += f"[{entity}]"
                current_entity = entity
            elif tag.startswith('I-') and current_entity == entity:
                continue
            else:
                current_entity = None
                masked += token

    return masked.strip()

marked_text = ner_nasking(ner_tag)
print(marked_text)

ร้องขอกฟ[ORGANIZATION][LOCATION][ORGANIZATION][LOCATION][LOCATION][LOCATION] เรื่องสถานะการขอขยายเขต ของบ้าน[LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION] ชื่อผู้ขอจดทะเบียนใช้ไฟฟ้า[PERSON]อินทร์ ผงทอง วันที่ยื่นเรื่อง[DATE]67 ผู้ใช้ไฟฟ้าแจ้งว่าได้ไปส่งเอกสารครบทั้งหมดแล้วเจ้าหน้าที่แจ้งว่างบประมาณหมด ต้องส่งเรื่องไป[ORGANIZATION] แต่เจ้าหน้าที่ยังไม่มีการส่งเรื่องไป [ORGANIZATION].ต้องให้[ORGANIZATION]ส่งเรื่องไปที่[ORGANIZATION]ให้ เนื่องจากต้องการใช้ไฟฟ้าด่วน รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //[PERSON]  หมายเลขติดต่อกลับ [PHONE]88


In [11]:
import re

def remove_redundant_tags(text):
    # ลบ tag ซ้ำติดกัน เช่น [ORG][ORG] เหลือ [ORG]
    return re.sub(r'(\[[A-Z]+\])\1+', r'\1', text)

In [12]:
text_ner = text_df.copy()

In [13]:
for i, row in text_ner.iterrows():
    ner_tag = get_ner_tag(row['text'])
    marked_text = ner_nasking(ner_tag)
    cleaned_text = remove_redundant_tags(marked_text)
    text_ner.at[i, 'marked_text'] = cleaned_text
    
text_ner.head(10)

RuntimeError: The expanded size of the tensor (803) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [1, 803].  Tensor sizes: [1, 512]

In [ ]:
text_ner.loc[9, 'marked_text']

'แนะนํา[ORGANIZATION][LOCATION]  เรื่องไฟฟ้าตกบ่อย [LOCATION]  ผู้ใช้ไฟฟ้าแจ้งว่าในพื้นที่มีกระแสไฟฟ้าตกบ่อยตกทุกวัน  [ORGANIZATION]ฟ.เป็นร้านขายยาจําเป็นต้องใช้แอร์ จากเหตุการณ์ที่เกิดขึ้นเกรงว่าอุปกรณ์เครื่องใช้ไฟฟ้าจะได้รับความเสียหาย จึงต้องการให้[ORGANIZATION] และหน่วยงานที่เกี่ยวข้องเข้าดําเนินการตรวจสอบปรับปรุงแก้ไขระบบคุณภาพไฟฟ้าในพื้นที่ให้ดียิ่งขึ้น//ผู้ใช้ไฟฟ้า[PERSON] หมายเลขติดต่อกลับ [PHONE]59'

In [ ]:
text_ner.to_csv('../data/processed/voc_data_marked_text_pea.csv', index=False, encoding='utf-8')

In [ ]:
text_ner.to_excel('../data/processed/voc_data_marked_text_pea.xlsx', index=False)